# 01 — Exploratory Data Analysis
Explore NYC/Boston 311 complaint patterns, hourly curves, and baseline distributions.

In [ ]:
import sys
sys.path.insert(0,'..')
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from src.data_ingestion import ingest_city_data
from src.preprocessing import clean, aggregate_time_windows, compute_rolling_baseline, city_level_aggregate
print('Ready')

## Load data

In [ ]:
nyc = ingest_city_data('nyc', days_back=45, force_synthetic=True)
bos = ingest_city_data('boston', days_back=45, force_synthetic=True)
print(f'NYC: {len(nyc):,}  Boston: {len(bos):,}')

## Category breakdown

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4))
for ax,(df,city) in zip(axes,[(nyc,'NYC'),(bos,'Boston')]):
    c=df['category'].value_counts()
    ax.barh(c.index,c.values,color='#1D4ED8',alpha=0.8)
    ax.set_title(f'{city} — by Category'); ax.set_xlabel('Count')
plt.tight_layout(); plt.show()

## Hourly patterns

In [ ]:
cl=clean(nyc); cl['hour']=cl['created_date'].dt.hour
hourly=cl.groupby(['hour','category']).size().unstack(fill_value=0)
fig,ax=plt.subplots(figsize=(11,4))
for col in hourly.columns: ax.plot(hourly.index,hourly[col],lw=2,label=col)
ax.set_xlabel('Hour'); ax.set_ylabel('Count'); ax.set_title('NYC — Hourly Volume by Category')
ax.legend(ncol=3); plt.tight_layout(); plt.show()

## Activity ratio distribution

In [ ]:
agg=aggregate_time_windows(cl,freq='h')
base=compute_rolling_baseline(agg,freq='h')
city_agg=city_level_aggregate(base,freq='h')
fig,ax=plt.subplots(figsize=(8,4))
for cat,color in zip(['noise','sanitation','housing'],['#1D4ED8','#15803D','#B91C1C']):
    sub=city_agg[city_agg['category']==cat]['activity_ratio'].clip(0,4)
    ax.hist(sub.dropna(),bins=50,alpha=0.5,color=color,label=cat,density=True)
ax.axvline(1.75,ls=':',lw=1.5,color='red',label='ACT threshold')
ax.set_xlabel('Activity ratio'); ax.legend(); ax.set_title('Activity Ratio Distributions')
plt.tight_layout(); plt.show()